In [2]:
import sklearn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

def train_val_test_split(test_ratio = 0.3,modalities = ["CTA", "MRA"], seed = 42):

    df = pd.read_csv('/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/train.csv')
    
    #print(df.head())

    df = df[df["Modality"].isin(modalities)]
    x = df["SeriesInstanceUID"]
    y = df["Aneurysm Present"] > 0
    x_train, x_test, y_train, y_test = train_test_split(x,y ,random_state=seed, test_size=test_ratio, shuffle=True,stratify = y)

    print("\n")
    print(f"[SPLIT] Train: {len(x_train)} | Test: {len(x_test)} | TOT: {len(x)}")
    print("\n")
    print(f"x = {x_train.iloc[0]} y = {y_train.iloc[0]}")
    return x_train,x_test,y_train,y_test



In [3]:
import os, warnings
import numpy as np, pandas as pd, pydicom
warnings.filterwarnings("ignore")

BASE_PATH = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/series"

TAGS = ["Modality", "Manufacturer", "ManufacturerModelName", "PatientSex", "PatientAge", "PatientWeight", "ScanOptions", "MRAcquisitionType", "SliceThickness", "RepetitionTime", "EchoTime", "PixelBandwidth", "ImagesInAcquisition", "PixelSpacing", "MagneticFieldStrength", "KVP"]

NUM = ["PatientAge", "PatientWeight", "SliceThickness", "RepetitionTime", "EchoTime", "PixelBandwidth", "ImagesInAcquisition", "PixelSpacing", "MagneticFieldStrength", "KVP"]
CAT = ["Modality", "Manufacturer", "ManufacturerModelName", "PatientSex", "ScanOptions", "MRAcquisitionType", "site_id"]
def extract_metadata(x_data):
    rows, index = [], []
    for idx, uid in x_data.items():
        folder = os.path.join(BASE_PATH, uid)
        files = os.listdir(folder) if os.path.isdir(folder) else []
        if not files:
            continue

        ds = pydicom.dcmread(os.path.join(folder, files[0]), stop_before_pixels=True)
        row = {}
        for t in TAGS:
            row[t] = str(ds.get(t, ""))
        row["site_id"] = str(ds[0x0013, 0x1001].value) if (0x0013, 0x1001) in ds else ""
        rows.append(row)
        index.append(idx)

    meta = pd.DataFrame(rows, index=index).replace(["", "None"], np.nan)
    meta["PatientAge"] = meta["PatientAge"].str.extract(r"(\d+)")[0]
    meta["PixelSpacing"] = meta["PixelSpacing"].str.extract(r"([\d.]+)")[0]
    #meta["vendor"] = meta["Manufacturer"].str.upper().str.split().str[0]

    for c in NUM:
        if c in meta:
            meta[c] = pd.to_numeric(meta[c], errors="coerce")
    for c in CAT:
        if c in meta:
            meta[c] = meta[c].astype("category")

    meta.to_csv("/kaggle/working/meta_series.csv", index=False)
    return meta[NUM + CAT]

#meta = extract_metadata(x_train)
#meta.head()

In [4]:
import numpy as np, pandas as pd, pydicom
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from xgboost import XGBClassifier


def train_xgboost(x_train, y_train,seed = 42):
    print("\nTRAIN\n")
    model = XGBClassifier(n_jobs=-1,random_state=seed,eval_metric="logloss", tree_method="hist", enable_categorical=True,)
    param_grid = {
        "n_estimators": [100, 200, 300],
        "max_depth": [2, 3, 4],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.7, 0.8, 1.0],
        "colsample_bytree": [0.7, 0.8, 1.0],
    }
    random_search = RandomizedSearchCV(model, param_grid, n_iter=30, cv=5,random_state=42,n_jobs=-1,scoring="roc_auc",)                                      
    
    random_search.fit(x_train, y_train)
    print("Best Params:", random_search.best_params_)
    print("CV AUC:", round(random_search.best_score_, 3))
    return random_search.best_estimator_, random_search.best_params_


def test_classifier(model, x_test, y_test):
    print("\nTEST\n")
    p = model.predict_proba(x_test)[:, 1]
    y_pred = (p >= 0.5).astype(int)
    print(f"AUC test: {roc_auc_score(y_test, p):.3f}")
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    return p




In [5]:
x_train,x_test,y_train,y_test = train_val_test_split()
x_train = extract_metadata(x_train)
x_test = extract_metadata(x_test)

y_train = y_train.loc[x_train.index]
y_test = y_test.loc[x_test.index]

for c in x_train.select_dtypes("category"):
    x_test[c] = x_test[c].cat.set_categories(x_train[c].cat.categories)


model , best_estimator =train_xgboost(x_train, y_train)
test_classifier(model, x_test, y_test)




[SPLIT] Train: 2142 | Test: 918 | TOT: 3060


x = 1.2.826.0.1.3680043.8.498.61470674983387614697021442480035027201 y = True

TRAIN

Best Params: {'subsample': 0.7, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.7}
CV AUC: 0.729

TEST

AUC test: 0.725

Classification Report:
               precision    recall  f1-score   support

       False       0.65      0.62      0.63       460
        True       0.64      0.67      0.65       458

    accuracy                           0.64       918
   macro avg       0.64      0.64      0.64       918
weighted avg       0.64      0.64      0.64       918

Confusion Matrix:
 [[284 176]
 [151 307]]


array([0.18033649, 0.5119041 , 0.7751178 , 0.7131136 , 0.41161552,
       0.42032838, 0.43794662, 0.8988348 , 0.6109717 , 0.59974706,
       0.5184467 , 0.12462476, 0.8861198 , 0.8181156 , 0.26796696,
       0.6078426 , 0.31718147, 0.4716661 , 0.7262774 , 0.41940176,
       0.5125556 , 0.1947998 , 0.37032235, 0.42032838, 0.39484766,
       0.09995859, 0.2873828 , 0.09011758, 0.09559514, 0.17635024,
       0.5579789 , 0.47427595, 0.48386988, 0.26835856, 0.52190214,
       0.2921426 , 0.57548696, 0.3890903 , 0.37514466, 0.42032838,
       0.50336266, 0.42032838, 0.755021  , 0.62032384, 0.19028339,
       0.7693984 , 0.46302217, 0.58060616, 0.41528577, 0.21852699,
       0.7658979 , 0.91056657, 0.19844782, 0.4942863 , 0.36664405,
       0.3817443 , 0.8743506 , 0.6381963 , 0.36664405, 0.7472443 ,
       0.9017221 , 0.21502732, 0.79622775, 0.8219721 , 0.60490996,
       0.17230806, 0.563285  , 0.9393294 , 0.16249567, 0.15574   ,
       0.7343435 , 0.62340975, 0.6109717 , 0.7970175 , 0.33071

In [6]:

import xgboost as xgb

shap = model.get_booster().predict(xgb.DMatrix(x_test, enable_categorical=True), pred_contribs=True)[:, :-1]
print(pd.Series(np.abs(shap).mean(0), index=x_test.columns).sort_values(ascending=False).round(3).to_string())

grupe = {
    "toate": list(x_train.columns),
    "fara varsta/sex": [c for c in x_train.columns if c not in ["PatientAge", "PatientSex"]],
    "doar varsta/sex": ["PatientAge", "PatientSex"],
    "doar Modality": ["Modality"],
    "doar site": ["site_id"],
    "fara varsta/sex/greutate": [c for c in x_train.columns if c not in ["PatientAge", "PatientSex", "PatientWeight"]],
}
print()
for nume, cols in grupe.items():
    m = XGBClassifier(**best_estimator, enable_categorical=True, eval_metric="logloss", random_state=42)
    m.fit(x_train[cols], y_train)
    print(f"{nume:18s} AUC test = {roc_auc_score(y_test, m.predict_proba(x_test[cols])[:, 1]):.3f}")

ManufacturerModelName    0.631
PatientAge               0.281
PatientSex               0.262
site_id                  0.257
ScanOptions              0.211
PixelSpacing             0.204
PatientWeight            0.167
KVP                      0.049
SliceThickness           0.047
PixelBandwidth           0.039
ImagesInAcquisition      0.029
Manufacturer             0.027
EchoTime                 0.026
RepetitionTime           0.026
MagneticFieldStrength    0.000
Modality                 0.000
MRAcquisitionType        0.000

toate              AUC test = 0.725
fara varsta/sex    AUC test = 0.705
doar varsta/sex    AUC test = 0.651
doar Modality      AUC test = 0.560
doar site          AUC test = 0.550
fara varsta/sex/greutate AUC test = 0.695


In [7]:
import os
import shap
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score

OUT_DIR = "/kaggle/working"
SHAP_EXPERIMENTS = ["doar CTA", "doar MRA", "CTA+MRA"]

def shap_explanation(model, x, out_dir, name):
    contribs = model.get_booster().predict(xgb.DMatrix(x, enable_categorical=True), pred_contribs=True)
    data = x.copy()
    for c in data.select_dtypes("category"):
        data[c] = data[c].cat.codes
    sv = shap.Explanation(values=contribs[:, :-1], base_values=contribs[:, -1], data=data.values, feature_names=list(x.columns))
    for plot_name, plot_fn in [("bar", shap.plots.bar), ("beeswarm", shap.plots.beeswarm)]:
        plot_fn(sv, max_display=10, show=False)
        plt.savefig(os.path.join(out_dir, f"shap_{plot_name}_{name}.png"), bbox_inches="tight")
        plt.close("all")

In [8]:
tests = {
    "toate": ["CTA", "MRA", "MRI T2", "MRI T1post"],
    "doar CTA": ["CTA"],
    "doar MRA": ["MRA"],
    "CTA+MRA": ["CTA", "MRA"],
}

SEEDS = [0,1,2,3,4]



def compute_metrics(y_test, p, multiclass=False):
    if multiclass:
        y_pred = p.argmax(axis=1)
        return {"auc_test": roc_auc_score(y_test, p, multi_class="ovr"),
                "accuracy": accuracy_score(y_test, y_pred),
                "balanced_acc": balanced_accuracy_score(y_test, y_pred),
                "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
                "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
                "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0)}
    y_pred = (p >= 0.5).astype(int)
    return {"auc_test": roc_auc_score(y_test, p),
            "accuracy": accuracy_score(y_test, y_pred),
            "balanced_acc": balanced_accuracy_score(y_test, y_pred),
            "sensitivity": recall_score(y_test, y_pred, zero_division=0),
            "specificity": recall_score(y_test, y_pred, pos_label=0, zero_division=0),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0)}




def run_xgb(include_site=False):
    rezultate = []
    for seed in SEEDS:
        for nume, mods in tests.items():
            print(f"\n{'=' * 15} {nume} | seed {seed} | site {include_site} {'=' * 15}")
            x_train, x_test, y_train, y_test = train_val_test_split(modalities=mods, seed=seed)
            x_train = extract_metadata(x_train)
            x_test = extract_metadata(x_test)
            y_train = y_train.loc[x_train.index]
            y_test = y_test.loc[x_test.index]

            x_train = x_train.dropna(axis=1, how="all")
            x_test = x_test[x_train.columns].copy()
            for c in x_train.select_dtypes("category"):
                x_test[c] = x_test[c].cat.set_categories(x_train[c].cat.categories)

            if not include_site:
                x_train = x_train.drop(columns=["site_id"])
                x_test = x_test.drop(columns=["site_id"])

            model, best_params = train_xgboost(x_train, y_train)
            p = test_classifier(model, x_test, y_test)
            metrics = compute_metrics(y_test.astype(int), p)

            if not include_site and nume in SHAP_EXPERIMENTS:
                out_dir = os.path.join(OUT_DIR, "shap", nume.replace(" ", "_"))
                os.makedirs(out_dir, exist_ok=True)
                shap_explanation(model, pd.concat([x_train, x_test]), out_dir, f"seed_{seed}")

            rezultate.append({"experiment": nume, "seed": seed, "n_train": len(x_train), "n_test": len(x_test), "pozitive_test": float(y_test.mean()), **metrics})
    return rezultate



# print("\nCU site_id")
# print(pd.DataFrame(rez_binary_classif_w_center).round(3).to_string(index=False))
# print("\nFARA site_id")
# print(pd.DataFrame(rez_binary_classif_no_center).round(3).to_string(index=False))

print()

rez_binary_classif_w_center = run_xgb(include_site=True)
rez_binary_classif_no_center = run_xgb(include_site=False)

NOT_METRICS = ["experiment", "seed", "n_train", "n_test", "pozitive_test"]

for titlu, rez, nume_fisier in [("W site_id", rez_binary_classif_w_center, "rsna_w_site"), ("NO site_id", rez_binary_classif_no_center, "rsna_no_site")]:
    df = pd.DataFrame(rez)
    metric_columns = [c for c in df.columns if c not in NOT_METRICS]
    summary = df.groupby("experiment")[metric_columns].agg(["mean", "std"]).round(3)

    print(f"\n{titlu} - all runs")
    print(df.round(3).to_string(index=False))
    print(f"\n{titlu} - avg(run(seeds))")
    print(summary.to_string())

    df.to_csv(f"{OUT_DIR}/{nume_fisier}_all_runs.csv", index=False)
    summary.to_csv(f"{OUT_DIR}/{nume_fisier}_summary.csv")





=============== toate | seed 0 | site True ===============


[SPLIT] Train: 3043 | Test: 1305 | TOT: 4348


x = 1.2.826.0.1.3680043.8.498.27406911615454951989336229409299052312 y = False

TRAIN

Best Params: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.7}
CV AUC: 0.739

TEST

AUC test: 0.769

Classification Report:
               precision    recall  f1-score   support

       False       0.73      0.80      0.76       746
        True       0.69      0.60      0.64       559

    accuracy                           0.71      1305
   macro avg       0.71      0.70      0.70      1305
weighted avg       0.71      0.71      0.71      1305

Confusion Matrix:
 [[598 148]
 [224 335]]

=============== doar CTA | seed 0 | site True ===============


[SPLIT] Train: 1265 | Test: 543 | TOT: 1808


x = 1.2.826.0.1.3680043.8.498.20166123858773468943663077834113252877 y = False

TRAIN

Best Params: {'subsample': 0.8, 'n_estimators': 100, 'max_

In [9]:
print("site_id" in x_train.columns)

True


In [10]:
!cd /kaggle/working && zip -r rsna_out.zip . -x ".virtual_documents/*"

  adding: rsna_w_site_summary.csv (deflated 58%)
  adding: rsna_no_site_summary.csv (deflated 59%)
  adding: rsna_w_site_all_runs.csv (deflated 58%)
  adding: rsna_no_site_all_runs.csv (deflated 60%)
  adding: meta_series.csv (deflated 87%)
  adding: shap/ (stored 0%)
  adding: shap/doar_MRA/ (stored 0%)
  adding: shap/doar_MRA/shap_beeswarm_seed_2.png (deflated 4%)
  adding: shap/doar_MRA/shap_bar_seed_2.png (deflated 15%)
  adding: shap/doar_MRA/shap_beeswarm_seed_0.png (deflated 5%)
  adding: shap/doar_MRA/shap_bar_seed_4.png (deflated 15%)
  adding: shap/doar_MRA/shap_beeswarm_seed_4.png (deflated 5%)
  adding: shap/doar_MRA/shap_bar_seed_1.png (deflated 15%)
  adding: shap/doar_MRA/shap_bar_seed_3.png (deflated 15%)
  adding: shap/doar_MRA/shap_beeswarm_seed_3.png (deflated 4%)
  adding: shap/doar_MRA/shap_beeswarm_seed_1.png (deflated 4%)
  adding: shap/doar_MRA/shap_bar_seed_0.png (deflated 16%)
  adding: shap/doar_CTA/ (stored 0%)
  adding: shap/doar_CTA/shap_beeswarm_seed_2.pn